In [20]:
import kagglehub
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

from catboost import CatBoostRegressor

import warnings
warnings.filterwarnings("ignore")

import os

train = pd.read_csv("/kaggle/input/datasets/mishtigarg/traffic/train.csv")
test = pd.read_csv("/kaggle/input/datasets/mishtigarg/traffic/test.csv")

def feature_engineering(df):

    df = df.copy()

    time_parts = (
        df["timestamp"]
        .astype(str)
        .str.split(":", expand=True)
    )

    df["hour"] = pd.to_numeric(
        time_parts[0],
        errors="coerce"
    )

    df["minute"] = pd.to_numeric(
        time_parts[1],
        errors="coerce"
    )

    df["hour"] = df["hour"].fillna(0)
    df["minute"] = df["minute"].fillna(0)

    df["is_morning_peak"] = (
        (df["hour"] >= 7) &
        (df["hour"] <= 10)
    ).astype(int)

    df["is_evening_peak"] = (
        (df["hour"] >= 17) &
        (df["hour"] <= 20)
    ).astype(int)

    df["road_weather"] = (
        df["RoadType"].astype(str)
        + "_"
        + df["Weather"].astype(str)
    )

    return df

train = feature_engineering(train)
test = feature_engineering(test)

cat_cols = [
    "geohash",
    "RoadType",
    "LargeVehicles",
    "Landmarks",
    "Weather",
    "road_weather"
]

for col in cat_cols:
    train[col] = train[col].fillna("Unknown")
    test[col] = test[col].fillna("Unknown")

num_cols = [
    "Temperature",
    "NumberofLanes"
]

for col in num_cols:
    median_val = train[col].median()

    train[col] = train[col].fillna(median_val)
    test[col] = test[col].fillna(median_val)

geo_mean = train.groupby("geohash")["demand"].mean()

global_mean = train["demand"].mean()

train["geo_mean"] = train["geohash"].map(geo_mean)
test["geo_mean"] = test["geohash"].map(geo_mean)

train["geo_mean"] = train["geo_mean"].fillna(global_mean)
test["geo_mean"] = test["geo_mean"].fillna(global_mean)


TARGET = "demand"

FEATURES = [
    "geohash",
    "day",
    "RoadType",
    "NumberofLanes",
    "LargeVehicles",
    "Landmarks",
    "Temperature",
    "Weather",
    "hour",
    "minute",
    "is_morning_peak",
    "is_evening_peak",
    "road_weather",
    "geo_mean"
]

X = train[FEATURES]
y = train[TARGET]

X_test = test[FEATURES]

cat_features = [
    FEATURES.index(col)
    for col in cat_cols
]

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof = np.zeros(len(train))
test_preds = np.zeros(len(test))

scores = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):

    print(f"Fold {fold+1}")

    X_train = X.iloc[tr_idx]
    y_train = y.iloc[tr_idx]

    X_valid = X.iloc[val_idx]
    y_valid = y.iloc[val_idx]

    model = CatBoostRegressor(
        iterations=3000,
        depth=8,
        learning_rate=0.03,
        loss_function="RMSE",
        random_seed=42,
        verbose=200
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        use_best_model=True
    )

    pred = model.predict(X_valid)

    score = r2_score(y_valid, pred)

    scores.append(score)

    print("R2 =", score)

    oof[val_idx] = pred

    test_preds += model.predict(X_test) / 5

print("Mean CV R2:", np.mean(scores))

submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": test_preds
})

submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()

from IPython.display import FileLink

FileLink("submission.csv")

Fold 1
0:	learn: 0.1385212	test: 0.1385341	best: 0.1385341 (0)	total: 51ms	remaining: 2m 32s
200:	learn: 0.0382829	test: 0.0394410	best: 0.0394410 (200)	total: 9.36s	remaining: 2m 10s
400:	learn: 0.0351304	test: 0.0368198	best: 0.0368198 (400)	total: 19.1s	remaining: 2m 3s
600:	learn: 0.0332290	test: 0.0354440	best: 0.0354440 (600)	total: 28.8s	remaining: 1m 55s
800:	learn: 0.0321434	test: 0.0347761	best: 0.0347761 (800)	total: 38.1s	remaining: 1m 44s
1000:	learn: 0.0311828	test: 0.0343083	best: 0.0343083 (1000)	total: 47.5s	remaining: 1m 34s
1200:	learn: 0.0303605	test: 0.0338824	best: 0.0338824 (1200)	total: 56.5s	remaining: 1m 24s
1400:	learn: 0.0296986	test: 0.0335634	best: 0.0335634 (1400)	total: 1m 5s	remaining: 1m 14s
1600:	learn: 0.0290934	test: 0.0333124	best: 0.0333124 (1600)	total: 1m 14s	remaining: 1m 5s
1800:	learn: 0.0286209	test: 0.0331774	best: 0.0331774 (1800)	total: 1m 23s	remaining: 55.9s
2000:	learn: 0.0282067	test: 0.0330297	best: 0.0330297 (1999)	total: 1m 33s	rem

/kaggle/working/submission.csv